# Stage 2 — SIM-level statistical forensic features

Build descriptive evidence features from canonical CDRs. This notebook does not assign a fraud score.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/statistical/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string','src_imei':'string','dst_imei':'string'},parse_dates=['timestamp'])
fraud=set(pd.read_csv(PROCESSED/'fraudulent_users.csv',dtype={'src_number':'string'})['src_number'].dropna().tolist())
print('CDRs:',cdr.shape,'fraud labels:',len(fraud))

CDRs: (86873, 18) fraud labels: 200


In [2]:
voice={'MOC','MTC','I-MOC','I-MTC'}; outgoing={'MOC','I-MOC','MO-SMS'}; intl={'I-MOC','I-MTC'}

src=cdr[['record_id','timestamp','cdr_type','src_number','dst_number','duration','current_src_cell_id','src_imei']].rename(columns={'src_number':'sim','dst_number':'contact','current_src_cell_id':'cell','src_imei':'imei'})
src['direction']='outgoing'; src['is_international']=src.cdr_type.isin(intl); src['is_voice']=src.cdr_type.isin(voice); src['is_outgoing']=src.cdr_type.isin(outgoing)
dst=cdr[['record_id','timestamp','cdr_type','dst_number','src_number','duration','current_dst_cell_id','dst_imei']].rename(columns={'dst_number':'sim','src_number':'contact','current_dst_cell_id':'cell','dst_imei':'imei'})
dst['direction']='incoming'; dst['is_international']=dst.cdr_type.isin(intl); dst['is_voice']=dst.cdr_type.isin(voice); dst['is_outgoing']=False
activity=pd.concat([src,dst],ignore_index=True).drop_duplicates(['sim','record_id'])
activity['known_fraud']=activity.sim.isin(fraud); activity['date']=activity.timestamp.dt.date; activity['hour']=activity.timestamp.dt.hour
print('SIM-event rows:',len(activity),'class counts:',activity[['sim','known_fraud']].drop_duplicates().known_fraud.value_counts().to_dict())

SIM-event rows: 98908 class counts: {False: 6793, True: 199}


In [3]:
features=activity.groupby('sim').agg(activity_count=('record_id','nunique'),total_duration=('duration','sum'),average_duration=('duration','mean'),median_duration=('duration','median'),maximum_duration=('duration','max'),minimum_duration=('duration','min'),active_days=('date','nunique'),active_hours=('hour','nunique'),unique_contacts=('contact','nunique'),unique_current_cells=('cell','nunique'),international_activity_count=('is_international','sum'),voice_activity_count=('is_voice','sum'),outgoing_activity_count=('is_outgoing','sum'),incoming_activity_count=('direction',lambda x:(x=='incoming').sum()),first_activity=('timestamp','min'),last_activity=('timestamp','max'),known_fraud=('known_fraud','max')).reset_index().rename(columns={'sim':'sim_number'})

def ratio(a,b): return a/b.replace(0,np.nan)
features['international_activity_ratio']=ratio(features.international_activity_count,features.activity_count)
features['voice_activity_ratio']=ratio(features.voice_activity_count,features.activity_count)
features['outgoing_to_incoming_ratio']=ratio(features.outgoing_activity_count,features.incoming_activity_count)
features['activity_span_seconds']=(features.last_activity-features.first_activity).dt.total_seconds()
features.to_csv(OUTPUT/'sim_statistical_features.csv',index=False)
features.head()

,sim_number,activity_count,total_duration,average_duration,median_duration,maximum_duration,minimum_duration,active_days,active_hours,unique_contacts,...,voice_activity_count,outgoing_activity_count,incoming_activity_count,first_activity,last_activity,known_fraud,international_activity_ratio,voice_activity_ratio,outgoing_to_incoming_ratio,activity_span_seconds
0,+11102241212,1,16,16.0,16.0,16,16,1,1,1,...,1,0,1,2021-03-17 15:27:40,2021-03-17 15:27:40,False,1.0,1.0,0.0,0.0
1,+11124222602,1,6,6.0,6.0,6,6,1,1,1,...,1,0,1,2021-03-17 11:35:28,2021-03-17 11:35:28,False,1.0,1.0,0.0,0.0
2,+11185236570,1,415,415.0,415.0,415,415,1,1,1,...,1,1,0,2021-03-17 13:14:58,2021-03-17 13:14:58,False,1.0,1.0,NaN,0.0
3,+11201660070,1,9,9.0,9.0,9,9,1,1,1,...,1,0,1,2021-03-17 14:22:26,2021-03-17 14:22:26,False,1.0,1.0,0.0,0.0
4,+11278389544,1,10,10.0,10.0,10,10,1,1,1,...,1,0,1,2021-03-17 19:17:47,2021-03-17 19:17:47,False,1.0,1.0,0.0,0.0


In [4]:
numeric=['activity_count','total_duration','average_duration','median_duration','maximum_duration','minimum_duration','active_days','active_hours','unique_contacts','unique_current_cells','international_activity_count','voice_activity_count','outgoing_activity_count','incoming_activity_count','international_activity_ratio','voice_activity_ratio','outgoing_to_incoming_ratio','activity_span_seconds']
for name,frame in [('all_sims',features),('known_fraudulent',features[features.known_fraud]),('known_legitimate',features[~features.known_fraud])]:
    table=frame[numeric].describe(percentiles=[.25,.5,.75]).T.rename(columns={'25%':'Q1','50%':'median','75%':'Q3'}); table['IQR']=table.Q3-table.Q1; table.to_csv(OUTPUT/f'sim_descriptive_statistics_{name}.csv')
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')
comparison=features.groupby('known_fraud',observed=False)[numeric].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']; comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'sim_group_median_comparison.csv'); features[numeric].corr(method='spearman').to_csv(OUTPUT/'sim_spearman_correlations.csv')
print('feature rows:',len(features)); print(comparison)

feature rows: 6992
                              known_legitimate_median  \
activity_count                               2.000000   
total_duration                              53.000000   
average_duration                            28.666667   
median_duration                             22.000000   
maximum_duration                            42.000000   
minimum_duration                             9.000000   
active_days                                  1.000000   
active_hours                                 1.000000   
unique_contacts                              1.000000   
unique_current_cells                         1.000000   
international_activity_count                 0.000000   
voice_activity_count                         1.000000   
outgoing_activity_count                      0.000000   
incoming_activity_count                      1.000000   
international_activity_ratio                 0.000000   
voice_activity_ratio                         1.000000   
outgoing_to_

Undefined ratios remain NaN. Group comparisons are descriptive evidence, not final fraud decisions.